In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

In [20]:
analysis_name = "gsa-shiva-mcf"

exp_params = {
    "phi0": 0.2,
    "reltime_offset": 0.0,
    "stride": 100
}

root_path = "../.."
model_folder_path=root_path+"/results/"+analysis_name+"/"
data_folder_path=root_path+"/../../data_repository/data/experiments/shiva_red_jupyter/"
comparisons_folder_path=root_path+"/comparetodata/plots/"+analysis_name+"/"

# metadata of experiments
table_path=root_path+"/../../data_repository/data/experiments/tables/"
dt={'event_id':'str','event_name':'str','s1949':'int','s1984_r1':'int','s1984_r2':'int','b1137':'int','s1994':'int','s1995':'int','s1996':'int','s1997':'int'}
indexes=pd.read_excel(table_path+'experiment_stages.xlsx',sheet_name='indexes',usecols=[0,1,2,3,4,5,6,7,8,9],dtype=dt)

# metadata of model
models_parameters_df = pd.read_excel(root_path+"/inputparameters.xlsx")
analysis_model_parameters = models_parameters_df[models_parameters_df["name"]==analysis_name]

# metadata of gsa
all_models_df = pd.read_csv(model_folder_path+"/"+analysis_name+"_analysis.csv")

# Filter for good models
good_models_df=all_models_df[all_models_df["goodmatch"]=="yes"]
# good_models_df=all_models_df[all_models_df["tinstability"].notna()]
# good_models_df=all_models_df[0:4]

In [ ]:
import numpy as np

def get_normalized_porosity(df,parameters):
    phi0 = parameters.get("phi0", 0.2)

    if "thickness_high" in df or ("thickness_high" in df and "LVDT_high" in df):

        # Method 1
        # h0 = df['thickness_high'].iloc[0]
        # df['normalized_porosity'] = (1 - phi0) * (1 - (h0 / df['thickness_high']))
        
        # Method 2
        print("Method of volume of solids and total volume (shiva)")
        volume_solids = 0.012 / 2400
        
        df['volume_solids'] = np.ones_like(df['thickness_high']) * volume_solids
        df["volume_total"] = (np.pi * (51.1/2000)**2) * df["thickness_high"]*1e-3
        df["porosity"] = 1 - (volume_solids / df["volume_total"])
        df["normalized_porosity"] = df["porosity"]-df["porosity"].iloc[0]

        # Method 3
        print("Method of Samuelson et al., 2009, equation 8 (shiva)")
        df["normalized_porosity_samuelson"] = df['delta_thickness_high'] / df['thickness_high'].iloc[0]

    elif ("LVDT_high" in df and "PC Displacement (mm)" not in df):
        print("Method of equivalent porosity for rocks (shiva)")

        initial_aperture = 0.0218 # mm - relative to P800 sandpaper
        df["aperture"] = -1* (df["LVDT_high"] - df["LVDT_high"].iloc[0]) + initial_aperture 
        df["porosity"] = df["aperture"] / 1 # reference of 1 mm
        df['normalized_porosity'] = df["porosity"] - df["porosity"].iloc[0]

    elif ('PC Displacement (mm)' in df and "LVDT_high" in df):
         # Method 2
        print("Method of volume of solids and total volume (brava)")
        volume_solids = 0.012 / 2600 #mass to be checked

        df["volume_total"] = (0.05*0.05) * (df["delta_LVDT_high"]*1e-3 + 0.003) # mm initial thickness to be checked
        df["porosity"] = 1 - (volume_solids / df["volume_total"])
        df["normalized_porosity"] = df["porosity"]-df["porosity"].iloc[0]

        # Method 3
        print("Method of Samuelson et al., 2009, equation 8 (shiva)")
        df["normalized_porosity_samuelson"] = df['delta_thickness_high'] / df['thickness_high'].iloc[0]

    return df

In [ ]:
import numpy as np

def mat2df(mat):
    # Filter only the relevant ndarrays and flatten them into a dict
    data = {k: v.flatten() for k, v in mat.items() if isinstance(v, np.ndarray)}
    return pd.DataFrame(data)

def slicer(exp,start,end,expname=None):
    
    a=indexes.loc[indexes['event_name']==start,expname].iloc[0].copy()
    b=indexes.loc[indexes['event_name']==end,expname].iloc[0].copy()

    #slice
    slexp_o = exp.iloc[a:b].copy()

    try:    
        th0=slexp_o['thickness_high'].iloc[0]
        slexp_o['delta_thickness_high']=slexp_o['thickness_high'].sub(th0)
    except:
        print('no thickness_high')

    try:
        th0=slexp_o['LVDT_high'].iloc[0]
        slexp_o['delta_LVDT_high']=slexp_o['LVDT_high'].sub(th0)
    except:
        print('no LVDT_high')

    try:
        th0=slexp_o['slip'].iloc[0]
        slexp_o['delta_slip']=slexp_o['slip'].sub(th0)
    except:
        print('no slip')

    c=indexes[indexes['event_name']==start].index.values
    d=indexes[indexes['event_name']==end].index.values
  
    slindexes_o=indexes[c[0]:d[0]]

    return slexp_o,slindexes_o


def experiment2normalized (df,parameters):

    v0 = parameters.get("v0", 1e-5)
    dc = parameters.get("dc", 3e-5)
    sigma0 = parameters.get("sigma0", 7.5)
    pf0 = parameters.get("pf0", 2.5)
    dpinjfactor = parameters.get("dpinjfactor", 1.0)
    
    df['normalized_time'] = df['Time_min']*60*v0/dc
    df['normalized_slip_rate'] = df['vel']/v0
    df['normalized_slip'] = df['slip'] / dc
    df['normalized_shear_stress'] = df['shear1'] / sigma0
    df['normalized_pore_pressure_fault'] = (df['Pf'] - pf0) / sigma0 * dpinjfactor
    df['normalized_pore_pressure_reservoir'] = (df['PumpPressure'] - pf0) / sigma0 * dpinjfactor

    # Normalized state

    V_init = df['vel'].iloc[0]

    # If steady state, Psi_0 = 1.0
    Psi_0 = 1.0

    s = df['slip'].values
    t = df['Time_min'].values * 60
    
    # Vectorize the differences and exponential factor
    ds = np.diff(s, prepend=s[0])
    dt = np.diff(t, prepend=t[0])
    exp_factor = np.exp(-ds / dc)
    
    # Pre-calculate v_local where possible to avoid division in the loop
    # We use np.divide to handle zeros safely
    v_local = np.divide(ds, dt, out=np.zeros_like(ds), where=dt!=0)
    
    psi = np.zeros(len(df))
    psi[0] = 1.0 # Psi_0
    
    for i in range(len(df) - 1):
        if dt[i+1] > 0 and ds[i+1] > 1e-6:
            psi[i+1] = psi[i] * exp_factor[i+1] + (v0 / v_local[i+1]) * (1 - exp_factor[i+1])
        else:
            psi[i+1] = psi[i] + (v0 / dc) * dt[i+1]

    df['normalized_state_variable'] = psi

    # Normalized porosity
    df=get_normalized_porosity(df,parameters)
    
    return df

In [23]:
def make_comparison_figure(df_model, df_experiment_normalized,stride=1,reltime_offset=0):
    %matplotlib qt

    # zeroing relative times and slip
    df_experiment_normalized['normalized_time']=df_experiment_normalized['normalized_time']-df_experiment_normalized['normalized_time'].iloc[0]+100
    df_model['time']=df_model['time']-df_model['time'].iloc[0]+100
    df_experiment_normalized['normalized_slip']=1e-9+df_experiment_normalized['normalized_slip']-df_experiment_normalized['normalized_slip'].iloc[0]
    df_model['U']=1e-9+df_model['U']-df_model['U'].iloc[0]

    fig, axs = plt.subplots(7, 1, figsize=(7, 16), sharex=True)

    experiment_style={'label': 'Experiment', 'color': 'black', 'linestyle': 'None', 'marker': 'o'}
    model_style={'label': 'Model', 'color': 'red', 'linestyle': '-'}

    axs[0].plot(df_experiment_normalized['normalized_time'][::stride]-reltime_offset, df_experiment_normalized['normalized_slip_rate'][::stride], **experiment_style)
    axs[0].plot(df_model['time'], df_model['V'], **model_style)
    axs[0].set_yscale('log')
    axs[0].set_ylabel('V')

    axs[1].plot(df_experiment_normalized['normalized_time'][::stride]-reltime_offset, df_experiment_normalized['normalized_state_variable'][::stride], **experiment_style)
    axs[1].plot(df_model['time'], df_model['Θ'], **model_style)
    axs[1].set_yscale('log')
    axs[1].set_ylabel('Θ')

    axs[2].plot(df_experiment_normalized['normalized_time'][::stride]-reltime_offset, df_experiment_normalized['normalized_slip'][::stride], **experiment_style)
    axs[2].plot(df_model['time'], df_model['U'], **model_style)
    axs[2].set_yscale('log')
    axs[2].set_ylabel('U')

    axs[3].plot(df_experiment_normalized['normalized_time'][::stride]-reltime_offset, df_experiment_normalized['normalized_shear_stress'][::stride] , **experiment_style)
    axs[3].plot(df_model['time'], df_model['Τ'], **model_style)
    axs[3].set_yscale('log')
    axs[3].set_ylabel('T')

    # axs[4].plot(df_experiment_normalized['normalized_time'][::stride]-reltime_offset, df_experiment_normalized['normalized_porosity'][::stride], **experiment_style)
    axs[4].plot(df_experiment_normalized['normalized_time'][::stride]-reltime_offset, df_experiment_normalized['normalized_porosity_samuelson'][::stride], **experiment_style)
    axs[4].plot(df_model['time'], df_model['Φ'], **model_style)
    # axs[4].set_yscale('log')
    axs[4].set_ylabel('Φ')

    axs[5].plot(df_experiment_normalized['normalized_time'][::stride]-reltime_offset, df_experiment_normalized['normalized_pore_pressure_fault'][::stride], **experiment_style)
    axs[5].plot(df_model['time'], df_model['P'], **model_style)
    # axs[5].set_yscale('log')
    axs[5].set_ylabel('P')

    axs[6].plot(df_experiment_normalized['normalized_time'][::stride]-reltime_offset, df_experiment_normalized['normalized_pore_pressure_reservoir'][::stride], **experiment_style)
    axs[6].plot(df_model['time'], df_model['Prev'], **model_style)
    # axs[6].set_yscale('log')
    axs[6].set_ylabel('Prev')

    # axs[6].set_xlim(-reltime_offset, None)
    return fig

In [24]:
# --- STEP 1: Pre-process experimental data (once) ---
exp_data_name = analysis_model_parameters["data"].values[0]
exp_params.update({
    "v0": analysis_model_parameters["v0 (µm/s)"].values[0]*1e-6,
    "dc": analysis_model_parameters["d_c (mm)"].values[0]*1e-3,
    "sigma0": analysis_model_parameters["sigma (MPa)"].values[0],
    "pf0": analysis_model_parameters["Pf (MPa)"].values[0],
    "dpinjfactor": analysis_model_parameters["dpinjfactor"].values[0]
})

# print(exp_params)

dict_experiment = sio.loadmat(f"{data_folder_path}/{exp_data_name}.mat")
df_exp_raw = mat2df(dict_experiment)
df_exp_sliced, _ = slicer(df_exp_raw, 'end preshear', 'end fast slip', expname=exp_data_name)
df_exp_norm = experiment2normalized(df_exp_sliced, exp_params)

# --- STEP 2: Loop through models efficiently ---
for row in good_models_df.itertuples():
    print(f"Processing index: {row.Index}")

    # Access values directly from 'row' instead of indexing the whole DF
    model_name = (f"{analysis_name}--difftime-{row.difftime:.2e}"
                  f"--alpha-{row.alpha:.2e}"
                  f"--epsilon-{row.epsilon:.2e}"
                  f"--beta-{row.beta:.2e}")

    # Load only the specific model results
    df_model = pd.read_csv(f"{model_folder_path}/{model_name}rt.csv")

    # Reuse the pre-processed experimental data
    figure = make_comparison_figure(df_model, df_exp_norm, stride=exp_params["stride"], reltime_offset=exp_params["reltime_offset"])
    figure.suptitle(f"Model: {model_name}")       

    figure.savefig(f"{comparisons_folder_path}/{model_name}_comparison.png")      

Method of volume of solids and total volume (shiva)
Method of Samuelson et al., 2009, equation 8 (shiva)
Processing index: 1
Processing index: 6
Processing index: 11
Processing index: 66
Processing index: 71
Processing index: 131


In [25]:
experiment_style={'color': 'black', 'linestyle': 'None', 'marker': 'o'}

if "aperture" in df_exp_norm:
    df_exp_norm.iloc[::exp_params["stride"]].plot(x="normalized_time", y=["LVDT_high","delta_LVDT_high","aperture", "porosity", "normalized_porosity"],subplots=True, **experiment_style)
elif "thickness_high" in df_exp_norm:
    df_exp_norm.iloc[::exp_params["stride"]].plot(x="normalized_time", y=["thickness_high","delta_thickness_high","volume_solids","volume_total","porosity", "normalized_porosity_samuelson"],subplots=True, **experiment_style)

In [26]:
df_exp_norm.keys()

Index(['TorqueLG', 'Axialload', 'Encoder1', 'Encoder2', 'LVDTlong',
       'LVDTshort', 'GEMSPressure', 'GefranPressure', 'AI6', 'AI7', 'AI16',
       'AI17', 'AI18', 'Stamp', 'Rate', 'RateZero', 'Time', 'Normal',
       'InternalTemperature', 'InternalPressure', 'PumpPressure',
       'BigMotorTorque', 'BigMotorShearStress', 'PumpVolume', 'GEM', 'Pf',
       'EffPressure', 'Pc', 'SlipVel_Enc_1', 'Slip_Enc_1', 'SlipVel_Enc_2',
       'Slip_Enc_2', 'slip', 'vel', 'LVDT_low', 'LVDT_high', 'shear1', 'Mu1',
       'TempE', 'thickness_low', 'thickness_high', 'AI6o', 'Time_hour',
       'Time_min', 'effstress', 'Mu_eff', 'Mu_app', 'delta_thickness_high',
       'delta_LVDT_high', 'delta_slip', 'normalized_time',
       'normalized_slip_rate', 'normalized_slip', 'normalized_shear_stress',
       'normalized_pore_pressure_fault', 'normalized_pore_pressure_reservoir',
       'normalized_state_variable', 'volume_solids', 'volume_total',
       'porosity', 'normalized_porosity', 'normalized_poros